In [ ]:
# Scaled Dot-Product Attention 标度点积注意力

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

def scaled_dot_product_attention(query, key, value, mask=None):
    """
    计算 Scaled Dot-Product Attention
    query: (batch_size, num_heads, seq_len_q, depth)
    key: (batch_size, num_heads, seq_len_k, depth)
    value: (batch_size, num_heads, seq_len_v, depth_v)
    mask: (batch_size, 1, seq_len_q, seq_len_k) or None
    """
    # 计算点积（矩阵乘法）
    matmul_qk = torch.matmul(query, key.transpose(-2, -1))  # (batch_size, num_heads, seq_len_q, seq_len_k)

    # 缩放
    dk = key.size()[-1]
    scores = matmul_qk / math.sqrt(dk)

    # 添加掩码（如果有）
    if mask is not None:
        scores += (mask * -1e9)  # 将掩码位置的值设为负无穷大

    # 计算注意力权重
    attention_weights = F.softmax(scores, dim=-1)  # (batch_size, num_heads, seq_len_q, seq_len_k)

    # 计算输出
    output = torch.matmul(attention_weights, value)  # (batch_size, num_heads, seq_len_q, depth_v)

    return output, attention_weights

MHA (Multi-Head Attention)

[《Attention Is All You Need》](https://arxiv.org/abs/1706.03762)

- 结构：每个头都有独立的 Q、K、V    （Q = K = V = H 头数）
- 公式：$$\text{head}_i = \text{Attention}(Q_i, K_i, V_i) \quad \forall i \in [1, H]$$
- 特点：表达能力最强，不同头可以学习完全不同的注意力模式（语法、语义、位置等）。

---

伪代码：
```
Q = X @ W_Q          # 每个头独立
K = X @ W_K
V = X @ W_V
把 Q/K/V 切成 H 个头
每个头独立做 Attention
拼接后 @ W_O
```

In [ ]:
# MHA (Multi-Head Attention) 多头注意力机制

class MHA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model // num_heads

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.w_out = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size = x.size(0)

        # 线性变换
        query = self.wq(x)  # (batch_size, seq_len, d_model)
        key = self.wk(x)    # (batch_size, seq_len, d_model)
        value = self.wv(x)  # (batch_size, seq_len, d_model)

        # 分割为多头
        query = query.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len, head_dim)
        key = key.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)      # (batch_size, num_heads, seq_len, head_dim)
        value = value.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)  # (batch_size, num_heads, seq_len, head_dim)

        # 计算注意力
        output, attention_weights = scaled_dot_product_attention(query, key, value, mask)

        # 合并多头
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)  # (batch_size, seq_len, d_model)

        return self.w_out(output)
        

MQA（Multi-Query Attention）

[《Fast Transformer Decoding: One Write-Head is All You Need》](https://arxiv.org/abs/1911.02150)

- 结构：所有头共享同一组 K 和 V（Q 头数 = H，K = V = 1 头数）
- 公式：$$\text{head}_i = \text{Attention}(Q_i, K_{\text{shared}}, V_{\text{shared}})$$
- 特点：KV Cache 最小（约为 MHA 的 $1/H$），推理最快，但表达能力下降最明显。

---

伪代码：
```
Q = X @ W_Q          # 多个头
K = X @ W_K          # 只有 1 个头
V = X @ W_V          # 只有 1 个头
把 K/V 在 head 维度上 repeat 到 H 次
做 Attention
```

In [ ]:
# MQA (Multi-Query Attention) 多查询注意力机制

class MQA(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model // num_heads

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, self.head_dim)
        self.wv = nn.Linear(d_model, self.head_dim)
        self.w_out = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, S, _ = x.shape

        q = self.wq(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, S, 1, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, S, 1, self.head_dim).transpose(1, 2)

        # repeat k and v for each head
        k = k.expand(-1, self.num_heads, -1, -1)
        v = v.expand(-1, self.num_heads, -1, -1)

        out = scaled_dot_product_attention(q, k, v, mask)
        out = out.transpose(1, 2).contiguous().view(B, S, -1)

        return self.w_out(out)

GQA (Grouped-Query Attention)

[《GQA: Training Generalized Multi-Query Transformer Models from Multi-Head Checkpoints》](https://arxiv.org/abs/2305.13245)

- 结构：把 $H$ 个 Query 头分成 $G$ 组（$1 < G < H$），每组内共享一组 K/V（Q 头数 = H，K = V = G 头数）
- 公式：$$\text{head}_i = \text{Attention}(Q_i, K_{g(i)}, V_{g(i)})$$
- 特点：MHA 和 MQA 的连续插值。当 $G = H$ 时就是 MHA，当 $G = 1$ 时就是 MQA。

---

伪代码：
```
Q = X @ W_Q                  # H 个头
K = X @ W_K                  # G 个头（G < H）
V = X @ W_V
把 K/V 按组 repeat_interleave 到 H 个头
做 Attention
```

In [ ]:
# GQA (Grouped Query Attention) 分组查询注意力机制

class GQA(nn.Module):
    def __init__(self, d_model, num_heads, num_kv_heads):
        super().__init__()
        assert num_heads % num_kv_heads == 0
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.num_queries_per_kv = num_heads // num_kv_heads
        self.head_dim = d_model // num_heads

        self.wq = nn.Linear(d_model, num_heads * self.head_dim)
        self.wk = nn.Linear(d_model, num_kv_heads * self.head_dim)
        self.wv = nn.Linear(d_model, num_kv_heads * self.head_dim)
        self.w_out = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, S, _ = x.shape

        q = self.wq(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, S, self.num_kv_heads, self.head_dim).transpose(1, 2)

        # repeat k and v for each group of queries
        k = k.repeat(1, self.num_queries_per_kv, 1, 1)
        v = v.repeat(1, self.num_queries_per_kv, 1, 1)

        out = scaled_dot_product_attention(q, k, v, mask)
        out = out.transpose(1, 2).contiguous().view(B, S, -1)

        return self.w_out(out)

假设 $H = 32$ 个头，序列长度 $L$，head dim $= D$：

| 维度                     | MHA                                           | GQA（G=8）                                  | MQA                                                 |
| ------------------------ | --------------------------------------------- | ------------------------------------------------- | --------------------------------------------------- |
| **KV 头数**              | 32                                            | 8                                                 | 1                                                   |
| **KV Cache大小**        | $2×L×32×D2$ | $2×L×8×D（25\%）$ | $2×L×1×D（约 3\%）$ |
| **参数量（注意力部分）** | $3HD^2$                                 | $(H + 2G)D^2$                       | $(H + 2)D^2$                            |
| **表达能力**             | 最强                                          | 接近 MHA                                          | 较弱                                                |
| **推理速度**             | 最慢                                          | 接近 MQA                                          | 最快                                                |
| **训练稳定性**           | 最好                                          | 好                                                | 可能不稳定                                          |

\

| 场景                        | 推荐方案                  | 原因与代表模型                                               |
| --------------------------- | ------------------------- | ------------------------------------------------------------ |
| **研究 / 训练阶段**         | **MHA**                   | 追求最高质量，参数量大没关系。早期 GPT-2/3、BERT、原始 Llama-1 |
| **中大型模型服务（主流）**  | **GQA**（常用 G=8） | 质量接近 MHA，显存和速度大幅改善。**Llama 2/3、Mistral、Qwen、Gemma** 等几乎全部采用 |
| **超大模型 / 极致推理速度** | **MQA**                   | 显存最小、延迟最低。PaLM、Falcon、StarCoder 等               |
| **长上下文（>8K）**         | GQA 或 MQA                | 显存压力极大，必须压缩 KV                                    |
| **边缘设备 / 实时系统**     | MQA                       | 显存和带宽极度受限                                           |
| **云端高吞吐服务**          | GQA-8                     | 质量和效率的最佳平衡点                                       |


MLA 是适用于 DeepSeek 的补充变体

In [ ]:
# MLA (Multi-head Latent Attention) -- 不常见

class MLA(nn.Module):
    def __init__(self, d_model, num_heads, kv_lora_rank, q_lora_rank=None):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.kv_lora_rank = kv_lora_rank
        self.q_lora_rank = q_lora_rank or d_model

        # KV 压缩 + 上投影
        self.w_dkv = nn.Linear(d_model, kv_lora_rank, bias=False)
        self.w_uk = nn.Linear(kv_lora_rank, num_heads * self.head_dim, bias=False)
        self.w_uv = nn.Linear(kv_lora_rank, num_heads * self.head_dim, bias=False)

        # Query（可低秩）
        self.w_dq = nn.Linear(d_model, self.q_lora_rank, bias=False)
        self.w_uq = nn.Linear(self.q_lora_rank, num_heads * self.head_dim, bias=False)

        self.w_out = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, S, _ = x.shape

        # 1. 压缩 KV（推理时只缓存这个）
        c_kv = self.w_dkv(x)                     # [B, S, kv_lora_rank]
        if kv_cache is not None:
            c_kv = torch.cat([kv_cache, c_kv], dim=1)
        
        # 2. 上投影恢复 K、V
        k = self.w_uk(c_kv).view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.w_uv(c_kv).view(B, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # 3. Query
        c_q = self.w_dq(x)
        q = self.w_uq(c_q).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        # 4. Attention
        out = scaled_dot_product_attention(q, k, v, mask)
        out = out.transpose(1, 2).contiguous().view(B, S, -1)
        return self.out_proj(out), c_kv   # 返回新的 cache